# 02 — Data Cleaning

Every cleaning step is **deterministic and row-wise** (no statistics learned from the data), so it is safe to apply before any split. Anything that requires learned statistics (imputation, scaling, encoding) lives inside the scikit-learn pipeline fitted on the training set only — see `src/data_preprocessing.py`.

In [1]:
import sys
sys.path.insert(0, 'C:\\Users\\OUISSAL\\Production-Ready Customer Churn')  # make 'src' importable
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import pandas as pd
from src.data_preprocessing import (
    clean_data, encode_target, load_raw_data,
)
from src.feature_engineering import add_features

df = load_raw_data()

## 1. The `TotalCharges` problem

In [2]:
blank = df['TotalCharges'].astype(str).str.strip().isin(['', 'nan'])
print(f'Rows with blank TotalCharges: {blank.sum()}')
df.loc[blank, ['tenure', 'MonthlyCharges', 'TotalCharges']]

Rows with blank TotalCharges: 11


,tenure,MonthlyCharges,TotalCharges
488,0,52.55,
753,0,20.25,
936,0,80.85,
1082,0,25.75,
1340,0,56.05,
3331,0,19.85,
3826,0,25.35,
4380,0,20.00,
5218,0,19.70,
6670,0,73.35,


## 2. Cleaning steps

1. strip column names;
2. empty strings -> NaN;
3. `TotalCharges` -> numeric (coerce);
4. `SeniorCitizen` 0/1 -> "No"/"Yes";
5. drop duplicates;
6. blank `TotalCharges` with `tenure == 0` -> `MonthlyCharges` (a brand-new customer has no billing history yet).

In [3]:
cleaned = clean_data(df)
print(f'Shape after cleaning : {cleaned.shape}')
print(f'Duplicates            : {cleaned.duplicated().sum()}')
print(f'Missing values        : {cleaned.isna().sum().sum()}')

Shape after cleaning : (7043, 21)
Duplicates            : 0
Missing values        : 0


## 3. Verification of the TotalCharges fix

In [4]:
new = cleaned[cleaned['tenure'] == 0]
assert new['TotalCharges'].notna().all()
assert (new['TotalCharges'] == new['MonthlyCharges']).all()
new[['tenure', 'MonthlyCharges', 'TotalCharges']]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,52.55
753,0,20.25,20.25
936,0,80.85,80.85
1082,0,25.75,25.75
1340,0,56.05,56.05
3331,0,19.85,19.85
3826,0,25.35,25.35
4380,0,20.00,20.00
5218,0,19.70,19.70
6670,0,73.35,73.35


## 4. Data types after cleaning

In [5]:
cleaned.dtypes.to_frame('dtype')

,dtype
customerID,str
gender,str
SeniorCitizen,str
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


## 5. Target encoding

In [6]:
y = encode_target(cleaned)
print(f'Churn rate: {y.mean():.1%}  (1 = churner)')

Churn rate: 26.5%  (1 = churner)


## 6. Feature engineering preview (details in notebook 03)

In [7]:
engineered = add_features(
    cleaned.drop(columns=['Churn', 'customerID'])
)
engineered[['tenure', 'tenure_group', 'num_services',
           'avg_monthly_spend']].head()

,tenure,tenure_group,num_services,avg_monthly_spend
0,1,0-12,1,29.850000
1,34,25-48,2,55.573529
2,2,0-12,2,54.075000
3,45,25-48,3,40.905556
4,2,0-12,0,75.825000


**Outcome** — the cleaned dataset has 7,043 rows, 0 duplicates, 0 missing values and a 26.5% churn rate, ready for EDA and modelling.